# Instacart Rules Mining with Apriori

This notebook mines frequent itemsets and association rules from the train_rules baskets.

## Goals
- Load prepared baskets from Notebook 4
- Build a sparse one-hot matrix
- Run Apriori with max itemset length = 3
- Generate association rules
- Keep recommendation-style rules (subset -> one item)
- Save itemsets and rules for evaluation

## Output
This notebook saves frequent itemsets and association rules in the outputs folder.

In [3]:
import os
import ast
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [4]:
def load_baskets_outputs():
    """
    Load basket tables from Notebook 4.
    """
    tables = {
        "baskets_train_rules": pd.read_csv("../outputs/baskets_train_rules_ready.csv"),
        "baskets_validation": pd.read_csv("../outputs/baskets_validation.csv"),
        "baskets_test": pd.read_csv("../outputs/baskets_test.csv"),
    }
    return tables


def parse_items_column(baskets_df):
    """
    Convert the items column from string to list.
    """
    df = baskets_df.copy()
    df["items"] = df["items"].apply(ast.literal_eval)
    return df


def basket_overview(baskets_df, name):
    """
    Return a simple basket summary.
    """
    row = {
        "dataset": name,
        "n_baskets": len(baskets_df),
        "n_users": baskets_df["user_id"].nunique() if "user_id" in baskets_df.columns else np.nan,
        "avg_basket_size": baskets_df["items"].apply(len).mean(),
        "median_basket_size": baskets_df["items"].apply(len).median(),
        "min_basket_size": baskets_df["items"].apply(len).min(),
        "max_basket_size": baskets_df["items"].apply(len).max(),
    }
    return pd.DataFrame([row])


def build_onehot_matrix(baskets_df):
    """
    Build a sparse one-hot matrix for Apriori.
    """
    te = TransactionEncoder()

    X_sparse = te.fit(baskets_df["items"]).transform(
        baskets_df["items"],
        sparse=True
    )

    col_names = [str(c) for c in te.columns_]
    X_df = pd.DataFrame.sparse.from_spmatrix(X_sparse, columns=col_names)

    return X_df, col_names


def sparse_info_from_onehot(X_df):
    """
    Compute matrix density and sparsity.
    """
    n_rows, n_cols = X_df.shape
    total_cells = n_rows * n_cols

    nnz = 0
    for col in X_df.columns:
        nnz += int(X_df[col].sparse.npoints)

    density = nnz / total_cells if total_cells > 0 else 0
    sparsity = 1 - density

    return pd.DataFrame([{
        "n_baskets": n_rows,
        "n_products": n_cols,
        "non_zero_values": nnz,
        "total_cells": total_cells,
        "density": density,
        "sparsity": sparsity,
    }])


def mine_frequent_itemsets_apriori(X_df, min_support=0.004, max_len=3):
    """
    Mine frequent itemsets with Apriori.
    """
    itemsets = apriori(
        X_df,
        min_support=min_support,
        use_colnames=True,
        max_len=max_len,
        low_memory=True,
    )
    itemsets = itemsets.sort_values("support", ascending=False).reset_index(drop=True)
    return itemsets


def add_itemset_metadata(itemsets):
    """
    Add itemset length and readable itemset string.
    """
    df = itemsets.copy()
    df["itemset_length"] = df["itemsets"].apply(len)
    df["itemsets_str"] = df["itemsets"].apply(
        lambda x: ", ".join(sorted(list(x)))
    )
    return df


def build_rules(itemsets, metric="confidence", min_threshold=0.10):
    """
    Build association rules from itemsets.
    """
    if len(itemsets) == 0:
        return pd.DataFrame()

    rules = association_rules(
        itemsets,
        metric=metric,
        min_threshold=min_threshold,
    )
    return rules


def add_rule_metadata(rules):
    """
    Add lengths and readable columns to rules.
    """
    if len(rules) == 0:
        return rules.copy()

    df = rules.copy()

    df["antecedent_len"] = df["antecedents"].apply(len)
    df["consequent_len"] = df["consequents"].apply(len)
    df["rule_len"] = df["antecedent_len"] + df["consequent_len"]

    df["antecedents_str"] = df["antecedents"].apply(
        lambda x: ", ".join(sorted(list(x)))
    )
    df["consequents_str"] = df["consequents"].apply(
        lambda x: ", ".join(sorted(list(x)))
    )

    df["rule_type"] = (
        df["antecedent_len"].astype(str)
        + "->"
        + df["consequent_len"].astype(str)
    )

    return df


def keep_single_item_consequents(rules):
    """
    Keep rules with one-item consequent.
    """
    if len(rules) == 0:
        return rules.copy()

    return rules[rules["consequent_len"] == 1].copy()


def sort_rules_for_recommendation(rules):
    """
    Sort rules for recommendation usage.
    """
    if len(rules) == 0:
        return rules.copy()

    df = rules.sort_values(
        ["confidence", "lift", "support"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    return df


def top_itemsets_by_length(itemsets, top_n=10):
    """
    Keep top itemsets by support inside each length.
    """
    if len(itemsets) == 0:
        return itemsets.copy()

    df = itemsets.copy()
    df["rank_in_length"] = df.groupby("itemset_length")["support"].rank(
        method="first",
        ascending=False,
    )
    df = df[df["rank_in_length"] <= top_n].copy()
    df = df.sort_values(["itemset_length", "rank_in_length"]).reset_index(drop=True)
    return df


def rules_summary(rules):
    """
    Build a compact summary of rules.
    """
    if len(rules) == 0:
        return pd.DataFrame([{
            "n_rules": 0,
            "avg_confidence": np.nan,
            "avg_lift": np.nan,
            "avg_support": np.nan,
            "max_rule_len": np.nan,
        }])

    return pd.DataFrame([{
        "n_rules": len(rules),
        "avg_confidence": rules["confidence"].mean(),
        "avg_lift": rules["lift"].mean(),
        "avg_support": rules["support"].mean(),
        "max_rule_len": rules["rule_len"].max(),
    }])


def itemsets_summary(itemsets):
    """
    Build a compact summary of itemsets.
    """
    if len(itemsets) == 0:
        return pd.DataFrame([{
            "n_itemsets": 0,
            "n_itemsets_len_1": 0,
            "n_itemsets_len_2": 0,
            "n_itemsets_len_3": 0,
            "max_itemset_len": 0,
        }])

    counts = itemsets["itemset_length"].value_counts().to_dict()

    return pd.DataFrame([{
        "n_itemsets": len(itemsets),
        "n_itemsets_len_1": int(counts.get(1, 0)),
        "n_itemsets_len_2": int(counts.get(2, 0)),
        "n_itemsets_len_3": int(counts.get(3, 0)),
        "max_itemset_len": int(itemsets["itemset_length"].max()),
    }])


def save_rules_outputs(itemsets, rules_all, rules_reco, prefix="apriori_train_rules"):
    """
    Save itemsets and rules to outputs.
    """
    os.makedirs("outputs", exist_ok=True)

    itemsets_to_save = itemsets.copy()
    if "itemsets" in itemsets_to_save.columns:
        itemsets_to_save["itemsets"] = itemsets_to_save["itemsets"].apply(
            lambda x: "|".join(sorted(list(x)))
        )

    rules_all_to_save = rules_all.copy()
    if len(rules_all_to_save) > 0:
        rules_all_to_save["antecedents"] = rules_all_to_save["antecedents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )
        rules_all_to_save["consequents"] = rules_all_to_save["consequents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )

    rules_reco_to_save = rules_reco.copy()
    if len(rules_reco_to_save) > 0:
        rules_reco_to_save["antecedents"] = rules_reco_to_save["antecedents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )
        rules_reco_to_save["consequents"] = rules_reco_to_save["consequents"].apply(
            lambda x: "|".join(sorted(list(x)))
        )

    itemsets_to_save.to_csv(
        f"../outputs/{prefix}_frequent_itemsets.csv",
        index=False,
    )
    rules_all_to_save.to_csv(
        f"../outputs/{prefix}_rules_all.csv",
        index=False,
    )
    rules_reco_to_save.to_csv(
        f"../outputs/{prefix}_rules_reco.csv",
        index=False,
    )

In [5]:
# Load basket tables
basket_tables = load_baskets_outputs()

baskets_train_rules = parse_items_column(basket_tables["baskets_train_rules"])
baskets_validation = parse_items_column(basket_tables["baskets_validation"])
baskets_test = parse_items_column(basket_tables["baskets_test"])

# Show basket summaries
display(basket_overview(baskets_train_rules, "train_rules_ready"))
display(basket_overview(baskets_validation, "validation"))
display(basket_overview(baskets_test, "test_final"))

,dataset,n_baskets,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size
0,train_rules_ready,150001,83209,8.065,7.0,2,75


,dataset,n_baskets,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size
0,validation,206209,206209,10.376792,9.0,1,121


,dataset,n_baskets,n_users,avg_basket_size,median_basket_size,min_basket_size,max_basket_size
0,test_final,131209,131209,10.552759,9.0,1,80


In [6]:
# Build sparse one-hot matrix on train_rules only
X_train_rules, product_columns = build_onehot_matrix(baskets_train_rules)

# Show sparse matrix info
display(sparse_info_from_onehot(X_train_rules))

C:\Users\tcham\AppData\Local\Temp\ipykernel_9148\4099410680.py:50: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  X_df = pd.DataFrame.sparse.from_spmatrix(X_sparse, columns=col_names)


,n_baskets,n_products,non_zero_values,total_cells,density,sparsity
0,150001,3000,1209758,450003000,0.002688,0.997312


In [7]:
# Set Apriori parameters
min_support_value = 0.004
max_itemset_len = 3

apriori_params = pd.DataFrame([{
    "min_support": min_support_value,
    "max_len": max_itemset_len,
    "n_products_in_matrix": len(product_columns),
    "n_baskets_in_matrix": len(baskets_train_rules),
}])
display(apriori_params)

,min_support,max_len,n_products_in_matrix,n_baskets_in_matrix
0,0.004,3,3000,150001


In [8]:
# Mine frequent itemsets
frequent_itemsets = mine_frequent_itemsets_apriori(
    X_train_rules,
    min_support=min_support_value,
    max_len=max_itemset_len,
)

frequent_itemsets = add_itemset_metadata(frequent_itemsets)

In [9]:
# Show itemset summaries
display(itemsets_summary(frequent_itemsets))
display(frequent_itemsets["itemset_length"].value_counts().sort_index())
display(top_itemsets_by_length(frequent_itemsets, top_n=10))

,n_itemsets,n_itemsets_len_1,n_itemsets_len_2,n_itemsets_len_3,max_itemset_len
0,587,403,181,3,3


itemset_length
1    403
2    181
3      3
Name: count, dtype: int64

,support,itemsets,itemset_length,itemsets_str,rank_in_length
0,0.166872,(24852),1,24852,1.0
1,0.132966,(13176),1,13176,2.0
2,0.092406,(21137),1,21137,3.0
3,0.084966,(21903),1,21903,4.0
4,0.075726,(47209),1,47209,5.0
5,0.061693,(47766),1,47766,6.0
6,0.052360,(47626),1,47626,7.0
7,0.050826,(16797),1,16797,8.0
8,0.049006,(26209),1,26209,9.0
9,0.048260,(27845),1,27845,10.0


In [10]:
# Build all rules
rules_all = build_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.10,
)

rules_all = add_rule_metadata(rules_all)

In [11]:
# Keep recommendation rules
rules_reco = keep_single_item_consequents(rules_all)
rules_reco = sort_rules_for_recommendation(rules_reco)

In [12]:
# Show rule summaries
display(rules_summary(rules_reco))

if len(rules_reco) > 0:
    display(rules_reco["rule_type"].value_counts())
    display(
        rules_reco[
            [
                "antecedents_str",
                "consequents_str",
                "support",
                "confidence",
                "lift",
                "antecedent_len",
                "consequent_len",
                "rule_len",
                "rule_type",
            ]
        ].head(30)
    )
else:
    display(pd.DataFrame({"message": ["No rules found with current thresholds"]}))

,n_rules,avg_confidence,avg_lift,avg_support,max_rule_len
0,209,0.202982,2.137759,0.007138,3


rule_type
1->1    200
2->1      9
Name: count, dtype: int64

,antecedents_str,consequents_str,support,confidence,lift,antecedent_len,consequent_len,rule_len,rule_type
0,41787,24852,0.004980,0.391304,2.344934,1,1,2,1->1
1,28204,24852,0.011907,0.374188,2.242363,1,1,2,1->1
2,"21137, 47209",13176,0.005440,0.363798,2.736031,2,1,3,2->1
3,"21903, 47209",13176,0.004393,0.361690,2.720177,2,1,3,2->1
4,45066,24852,0.010253,0.359766,2.155938,1,1,2,1->1
5,8424,24852,0.005080,0.347628,2.083197,1,1,2,1->1
6,"21137, 27966",13176,0.004147,0.343836,2.585901,2,1,3,2->1
7,49683,24852,0.011927,0.339211,2.032760,1,1,2,1->1
8,9387,24852,0.004027,0.327727,1.963937,1,1,2,1->1
9,8174,13176,0.005013,0.326247,2.453618,1,1,2,1->1


In [13]:
# Show top 2->1 rules if available
rules_2_to_1 = rules_reco[rules_reco["rule_type"] == "2->1"].copy()
display(
    rules_2_to_1[
        [
            "antecedents_str",
            "consequents_str",
            "support",
            "confidence",
            "lift",
            "rule_type",
        ]
    ].head(20)
)

,antecedents_str,consequents_str,support,confidence,lift,rule_type
2,"21137, 47209",13176,0.005440,0.363798,2.736031,2->1
3,"21903, 47209",13176,0.004393,0.361690,2.720177,2->1
6,"21137, 27966",13176,0.004147,0.343836,2.585901,2->1
23,"13176, 27966",21137,0.004147,0.289437,3.132229,2->1
43,"13176, 21137",47209,0.005440,0.251464,3.320699,2->1
45,"13176, 21903",47209,0.004393,0.249244,3.291380,2->1
51,"13176, 47209",21137,0.005440,0.244092,2.641517,2->1
101,"13176, 47209",21903,0.004393,0.197128,2.320082,2->1
105,"13176, 21137",27966,0.004147,0.191680,3.987258,2->1


In [14]:
# Save outputs
save_rules_outputs(
    itemsets=frequent_itemsets,
    rules_all=rules_all,
    rules_reco=rules_reco,
    prefix="apriori_train_rules",
)

In [15]:
# notebook summary
final_summary = pd.DataFrame([{
    "n_frequent_itemsets": len(frequent_itemsets),
    "n_rules_all": len(rules_all),
    "n_rules_reco": len(rules_reco),
    "n_rules_1_to_1": int((rules_reco["rule_type"] == "1->1").sum()) if len(rules_reco) > 0 else 0,
    "n_rules_2_to_1": int((rules_reco["rule_type"] == "2->1").sum()) if len(rules_reco) > 0 else 0,
    "max_itemset_len_found": int(frequent_itemsets["itemset_length"].max()) if len(frequent_itemsets) > 0 else 0,
    "outputs_saved": True,
}])
display(final_summary)

,n_frequent_itemsets,n_rules_all,n_rules_reco,n_rules_1_to_1,n_rules_2_to_1,max_itemset_len_found,outputs_saved
0,587,209,209,200,9,3,True
